# Customer Lifetime Value Analysis
**Vintage Sampling Media Resale | Jan 2025 – Jun 2026**

This notebook estimates Customer Lifetime Value (CLV) using three approaches of increasing sophistication:

1. **Simple Historical CLV** — average order value × purchase frequency × lifespan
2. **Retention-Based CLV** — discounted cash flow model using estimated churn rate
3. **BG/NBD Model** — probabilistic model for non-contractual purchase behavior

The dataset covers 18 months of transaction data from a vintage sampling media resale business.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from lifetimes import BetaGeoFitter
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix
from lifetimes.utils import summary_data_from_transaction_data

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load cleaned order-level data
df = pd.read_csv('../data/orders_clean.csv', parse_dates=['order_date'])

print(f'Orders: {len(df):,}')
print(f'Unique buyers: {df["buyer_id"].nunique():,}')
print(f'Date range: {df["order_date"].min().date()} to {df["order_date"].max().date()}')
print(f'\nOrder value summary:')
print(df['order_value'].describe())

## 2. Simple Historical CLV

The most straightforward approach: estimate CLV from observed behavior.

$$CLV_{simple} = \bar{AOV} \times f \times L$$

Where:
- $\bar{AOV}$ = average order value per customer
- $f$ = average purchase frequency (orders per month)
- $L$ = average customer lifespan in months (first to last purchase)

In [ ]:
# Compute per-customer metrics
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

customer_df = df.groupby('buyer_id').agg(
    first_order   = ('order_date',  'min'),
    last_order    = ('order_date',  'max'),
    n_orders      = ('order_date',  'count'),
    total_revenue = ('order_value', 'sum'),
    avg_order_value = ('order_value', 'mean'),
).reset_index()

# Lifespan in months (at least 1 month for single-purchase customers)
customer_df['lifespan_months'] = (
    (customer_df['last_order'] - customer_df['first_order']).dt.days / 30.44
).clip(lower=1)

# Purchase frequency: orders per month of lifespan
customer_df['freq_per_month'] = customer_df['n_orders'] / customer_df['lifespan_months']

# Simple historical CLV
customer_df['clv_simple'] = (
    customer_df['avg_order_value'] *
    customer_df['freq_per_month'] *
    customer_df['lifespan_months']
)

print('Per-customer metrics:')
print(customer_df[['n_orders','avg_order_value','lifespan_months','clv_simple']].describe())

In [ ]:
# Segment customers by order count
customer_df['segment'] = pd.cut(
    customer_df['n_orders'],
    bins=[0, 1, 3, 10, np.inf],
    labels=['One-time', 'Occasional (2-3)', 'Regular (4-10)', 'VIP (10+)']
)

seg_summary = customer_df.groupby('segment', observed=True).agg(
    n_customers   = ('buyer_id',        'count'),
    avg_clv       = ('clv_simple',      'mean'),
    avg_orders    = ('n_orders',        'mean'),
    avg_aov       = ('avg_order_value', 'mean'),
    total_revenue = ('total_revenue',   'sum'),
).round(2)

print('\nCLV by customer segment:')
print(seg_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CLV distribution
axes[0].hist(customer_df['clv_simple'].clip(upper=customer_df['clv_simple'].quantile(0.95)),
             bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Simple CLV Distribution (95th pct cap)')
axes[0].set_xlabel('CLV ($)')
axes[0].set_ylabel('Customers')

# Revenue by segment
seg_summary['total_revenue'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Total Revenue by Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/clv_simple.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Retention-Based CLV

A more principled approach that accounts for the time value of money and customer churn.

$$CLV_{retention} = \frac{m \cdot r}{1 + d - r}$$

Where:
- $m$ = average margin per period (approximated by average order value)
- $r$ = retention rate (1 - churn rate)
- $d$ = discount rate (monthly)

We estimate churn as customers who haven't purchased in the last 90 days.

In [ ]:
# Estimate churn rate
cutoff = snapshot_date - pd.Timedelta(days=90)

repeat_buyers = customer_df[customer_df['n_orders'] > 1]
churned = repeat_buyers[repeat_buyers['last_order'] < cutoff]

churn_rate   = len(churned) / len(repeat_buyers) if len(repeat_buyers) > 0 else 0.5
retention_rate = 1 - churn_rate

print(f'Repeat buyers:    {len(repeat_buyers)}')
print(f'Churned buyers:   {len(churned)}')
print(f'Churn rate:       {churn_rate:.1%}')
print(f'Retention rate:   {retention_rate:.1%}')

In [ ]:
# Retention-based CLV
avg_order_value = customer_df['avg_order_value'].mean()
discount_rate   = 0.01  # 1% monthly (~12% annual)

clv_retention = (avg_order_value * retention_rate) / (1 + discount_rate - retention_rate)

print(f'Average order value:       ${avg_order_value:.2f}')
print(f'Discount rate (monthly):   {discount_rate:.1%}')
print(f'Retention-based CLV:       ${clv_retention:.2f}')

# Sensitivity analysis — how CLV changes with retention rate
retention_range = np.arange(0.1, 1.0, 0.05)
clv_range = (avg_order_value * retention_range) / (1 + discount_rate - retention_range)

plt.figure(figsize=(8, 4))
plt.plot(retention_range, clv_range, color='steelblue', linewidth=2)
plt.axvline(retention_rate, color='firebrick', linestyle='--', label=f'Observed retention: {retention_rate:.0%}')
plt.axhline(clv_retention, color='firebrick', linestyle=':')
plt.title('CLV Sensitivity to Retention Rate')
plt.xlabel('Retention Rate')
plt.ylabel('CLV ($)')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/clv_retention_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. BG/NBD Model (Probabilistic CLV)

The Beta-Geometric / Negative Binomial Distribution model is the gold standard for non-contractual customer settings (eCommerce, resale marketplaces) where customers don't formally cancel.

It models two simultaneous processes:
- **Purchase process (NBD):** how often a customer transacts while active
- **Dropout process (Beta-Geometric):** when a customer permanently churns

Inputs are the standard RFM summary:
- **frequency:** number of repeat purchases
- **recency:** time between first and last purchase
- **T:** total observation time since first purchase

In [ ]:
# Build RFM summary for BG/NBD
rfm = summary_data_from_transaction_data(
    df,
    customer_id_col='buyer_id',
    datetime_col='order_date',
    monetary_value_col='order_value',
    observation_period_end=snapshot_date,
    freq='D'
)

print('RFM summary shape:', rfm.shape)
print(rfm.describe())

In [ ]:
# Fit BG/NBD model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])

print('BG/NBD model fitted successfully')
print(bgf.summary)

1. Frequency histogram: actual vs predicted

Does the model reproduce the observed purchase frequency distribution?

In [ ]:
from lifetimes.plotting import plot_period_transactions

plt.figure(figsize=(8, 4))
plot_period_transactions(bgf)
plt.title('Actual vs Predicted Purchase Frequency')
plt.tight_layout()
plt.savefig('../outputs/bgnbd_freq_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

2. Calibration check — predicted vs actual for holdout period

Split data into calibration and holdout periods, see if predictions match:

In [ ]:
from lifetimes.utils import calibration_and_holdout_data

summary_cal_holdout = calibration_and_holdout_data(
    df,
    customer_id_col='buyer_id',
    datetime_col='order_date',
    calibration_period_end='2026-01-01',
    observation_period_end=snapshot_date,
    freq='D'
)

bgf_cal = BetaGeoFitter(penalizer_coef=0.01)
bgf_cal.fit(
    summary_cal_holdout['frequency_cal'],
    summary_cal_holdout['recency_cal'],
    summary_cal_holdout['T_cal']
)

from lifetimes.plotting import plot_calibration_purchases_vs_holdout_purchases

plt.figure(figsize=(8, 4))
plot_calibration_purchases_vs_holdout_purchases(bgf_cal, summary_cal_holdout)
plt.title('Calibration vs Holdout: Predicted vs Actual')
plt.tight_layout()
plt.savefig('../outputs/bgnbd_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Predict expected purchases in next 90 days
rfm['predicted_purchases_90d'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90, rfm['frequency'], rfm['recency'], rfm['T']
)

# Probability customer is still alive
rfm['prob_alive'] = bgf.conditional_probability_alive(
    rfm['frequency'], rfm['recency'], rfm['T']
)

# Estimated CLV = predicted purchases × average order value
rfm['clv_bgnbd_90d'] = rfm['predicted_purchases_90d'] * avg_order_value

print('Top 10 customers by predicted CLV (next 90 days):')
print(rfm.nlargest(10, 'clv_bgnbd_90d')[['frequency','recency','T','prob_alive','clv_bgnbd_90d']])

3. Model parameters

Check if the fitted parameters make business sense:

In [ ]:
print('BG/NBD Model Parameters:')
print(f'r (purchase rate shape):     {bgf.params_["r"]:.4f}')
print(f'alpha (purchase rate scale): {bgf.params_["alpha"]:.4f}')
print(f'a (dropout shape):           {bgf.params_["a"]:.4f}')
print(f'b (dropout scale):           {bgf.params_["b"]:.4f}')
print(f'\nImplied avg purchases/period: {bgf.params_["r"]/bgf.params_["alpha"]:.4f}')
print(f'Implied avg dropout prob:     {bgf.params_["a"]/(bgf.params_["a"]+bgf.params_["b"]):.4f}')

Frequency-Recency Matrix

In [ ]:
# Plot 1
plt.figure(figsize=(7, 5))
plot_frequency_recency_matrix(bgf)
plt.title('Expected Purchases (Next 90 Days)\nby Frequency & Recency')
plt.tight_layout()
plt.savefig('../outputs/bgnbd_frequency_recency.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2
plt.figure(figsize=(7, 5))
plot_probability_alive_matrix(bgf)
plt.title('Probability Customer is Still Active\nby Frequency & Recency')
plt.tight_layout()
plt.savefig('../outputs/bgnbd_prob_alive.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpreting the Frequency-Recency Matrix

**Important:** In the BG/NBD framework, "recency" is defined as the number of days 
between a customer's *first* and *last* purchase — not days since their last purchase. 

This means:
- **Higher recency value** → last purchase was further into the customer's history → more recent relative to observation date
- **Lower recency value** → customer made their last purchase soon after their first purchase and hasn't been seen since → likely churned

The hot zone (bottom-right) represents the highest-value customers:
- **High frequency** — many repeat purchases
- **High recency** — last purchase was recent relative to their customer lifetime

These are your most engaged, highest predicted future value customers.

## 5. CLV Model Comparison

Summary of the three approaches and their appropriate use cases.

In [ ]:
summary = pd.DataFrame({
    'Method': ['Simple Historical', 'Retention-Based', 'BG/NBD (90-day)'],
    'Avg CLV': [
        customer_df['clv_simple'].mean(),
        clv_retention,
        rfm['clv_bgnbd_90d'].mean()
    ],
    'Use Case': [
        'Quick baseline, easy to explain',
        'Subscription or repeat purchase businesses',
        'Non-contractual eCommerce (recommended)'
    ]
})

summary['Avg CLV'] = summary['Avg CLV'].map('${:.2f}'.format)
print(summary.to_string(index=False))

In [ ]:
print(rfm['clv_bgnbd_90d'].describe())
print(rfm.nlargest(10, 'clv_bgnbd_90d')[['frequency','prob_alive','clv_bgnbd_90d']])

In [ ]:
print(f"Top 10% of buyers account for {rfm.nlargest(int(len(rfm)*0.1), 'clv_bgnbd_90d')['clv_bgnbd_90d'].sum() / rfm['clv_bgnbd_90d'].sum():.1%} of predicted 90-day revenue")

In [ ]:
# Save enriched customer table
rfm.to_csv('../data/clv_output.csv')
print('Saved: clv_output.csv')

## Key Takeaways

- The majority of buyers are **one-time purchasers** typical for a niche resale market
- A small segment of **repeat buyers** drives disproportionate revenue
- The **BG/NBD model** identifies which repeat buyers are most likely still active vs churned, enabling targeted re-engagement
- CLV estimates inform how much to invest in customer acquisition and retention relative to expected return